In [ ]:
import json
from src.predict import load_model, predict
import requests

In [ ]:
model_path = "fc_model_weights.pth"
image_path = "data/test-data/1.jpeg"
benefits_path = "fruit_benefits.json"

In [ ]:
model = load_model(model_path)
fruit_name, confidence = predict(model, image_path)
print(f"Predicted: {fruit_name} ({(100*confidence):>0.1f}%)")

In [ ]:
with open(benefits_path) as f:
    fruit_benefits = json.load(f)

retrieved_info = fruit_benefits[fruit_name]

In [ ]:
url = "http://localhost:11434/api/generate"
llm_data = {
    "model": "gemma4:e2b",
    "stream": True,
    "prompt": f"Summarize the benefits and nutritional value of {fruit_name} using the following information. Benefits: {retrieved_info['benefits']}, Nutrients: {retrieved_info['nutrients']}"
}

# stream=True keeps the connection ope
with requests.post(url, json=llm_data, stream=True) as response:
    response.raise_for_status()
    
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line.decode('utf-8'))
            print(chunk['response'], end='', flush=True)
